In [2]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
PROJECT_ROOT = Path("../../").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "biomarkers"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "biomarkers"
PREPROCESSOR_DIR = PROJECT_ROOT / "artifacts" / "preprocessors" / "biomarkers"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSOR_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Preprocessors:", PREPROCESSOR_DIR)

Raw data: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\biomarkers
Processed data: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\biomarkers
Preprocessors: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\preprocessors\biomarkers


#### ZHEEN

In [4]:
DATASETS = {
    "zheen": RAW_DIR / "zheen",
    "uci_heart_failure": RAW_DIR / "uci_heart_failure",
    "mi_complications": RAW_DIR / "mi_complications",
    "framingham": RAW_DIR / "framingham",
}

for name, path in DATASETS.items():
    print(f"{name:25} -> {path.exists()} | {path}")

zheen                     -> True | D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\biomarkers\zheen
uci_heart_failure         -> True | D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\biomarkers\uci_heart_failure
mi_complications          -> True | D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\biomarkers\mi_complications
framingham                -> True | D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\biomarkers\framingham


In [5]:
zheen_files = list(DATASETS["zheen"].glob("*.csv"))

if not zheen_files:
    raise FileNotFoundError("No Zheen CSV found.")

zheen_raw = pd.read_csv(zheen_files[0])

print("Zheen shape:", zheen_raw.shape)
print("Columns:", zheen_raw.columns.tolist())

Zheen shape: (1319, 9)
Columns: ['Age', 'Gender', 'Heart rate', 'Systolic blood pressure', 'Diastolic blood pressure', 'Blood sugar', 'CK-MB', 'Troponin', 'Result']


In [6]:
zheen = zheen_raw.rename(columns={
    "Result": "target",
    "AGE": "age",
    "Age": "age",
    "Gender": "gender",
    "Heart rate": "heart_rate",
    "Systolic blood pressure": "systolic_bp",
    "Diastolic blood pressure": "diastolic_bp",
    "Blood sugar": "blood_glucose",
    "CK-MB": "ck_mb",
    "Troponin": "troponin",
}).copy()

required_columns = [
    "age",
    "gender",
    "heart_rate",
    "systolic_bp",
    "diastolic_bp",
    "blood_glucose",
    "ck_mb",
    "troponin",
    "target",
]

missing = [
    col for col in required_columns
    if col not in zheen.columns
]

if missing:
    raise ValueError(f"Missing Zheen columns: {missing}")

print("Zheen canonical schema validated.")
display(zheen[required_columns].head())

Zheen canonical schema validated.


,age,gender,heart_rate,systolic_bp,diastolic_bp,blood_glucose,ck_mb,troponin,target
0,63,1,66,160,83,160.0,1.80,0.012,negative
1,20,1,94,98,46,296.0,6.75,1.060,positive
2,56,1,64,160,77,270.0,1.99,0.003,negative
3,66,1,70,120,55,270.0,13.87,0.122,positive
4,54,1,64,112,65,300.0,1.08,0.003,negative


In [7]:
zheen["target"] = (
    zheen["target"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "positive": 1,
        "negative": 0,
    })
)

if zheen["target"].isna().any():
    raise ValueError("Unexpected Zheen target values found.")

print(zheen["target"].value_counts())

target
1    810
0    509
Name: count, dtype: int64


#### UCI HEART FAILURE

In [8]:
uci_files = list(DATASETS["uci_heart_failure"].glob("*.csv"))

if not uci_files:
    raise FileNotFoundError("No UCI Heart Failure CSV found.")

uci_hf = pd.read_csv(uci_files[0])

print("Original shape:", uci_hf.shape)

if "time" in uci_hf.columns:
    uci_hf = uci_hf.drop(columns=["time"])

print("Shape after leakage removal:", uci_hf.shape)

display(uci_hf.head())

Original shape: (299, 13)
Shape after leakage removal: (299, 12)


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,1


In [9]:
if "DEATH_EVENT" not in uci_hf.columns:
    raise KeyError("DEATH_EVENT not found in UCI Heart Failure dataset.")

uci_hf["DEATH_EVENT"] = uci_hf["DEATH_EVENT"].astype(int)

print(uci_hf["DEATH_EVENT"].value_counts())

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64


#### MI COMPLIPICATIONS

In [10]:
mi_files = list(DATASETS["mi_complications"].glob("*.csv"))

if not mi_files:
    raise FileNotFoundError("No MI-Complications CSV found.")

mi = pd.read_csv(mi_files[0])

print("MI-Complications shape:", mi.shape)
display(mi.head())

MI-Complications shape: (1700, 124)


,ID,AGE,SEX,INF_ANAM,STENOK_AN,FK_STENOK,IBS_POST,IBS_NASL,GB,SIM_GIPERT,...,JELUD_TAH,FIBR_JELUD,A_V_BLOK,OTEK_LANC,RAZRIV,DRESSLER,ZSN,REC_IM,P_IM_STEN,LET_IS
0,1,77.0,1,2.0,1.0,1.0,2.0,NaN,3.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2,55.0,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,3,52.0,1,0.0,0.0,0.0,2.0,NaN,2.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,4,68.0,0,0.0,0.0,0.0,2.0,NaN,2.0,0.0,...,0,0,0,0,0,0,1,0,0,0
4,5,60.0,1,0.0,0.0,0.0,2.0,NaN,3.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
cpk_columns = [
    col for col in mi.columns
    if "CPK" in col.upper()
]

print("CPK-like columns:", cpk_columns)

for col in cpk_columns:
    missing_pct = mi[col].isna().mean() * 100
    print(f"{col}: {missing_pct:.2f}% missing")

    if missing_pct >= 95:
        mi = mi.drop(columns=[col])

print("Shape after unusable-column removal:", mi.shape)

CPK-like columns: []
Shape after unusable-column removal: (1700, 124)


In [12]:
MI_BINARY_TARGETS = [
    "FIBR_PREDS",   # AF
    "PREDS_TAH",    # SVT
    "JELUD_TAH",    # VT
    "FIBR_JELUD",   # VF
    "A_V_BLOK",     # 3rd degree AV block
    "OTEC_LANC",    # pulmonary edema
    "RAZRIV",       # myocardial rupture
    "DRESSLER",     # Dressler syndrome
    "ZSN",          # chronic HF
    "REC_IM",       # recurrent MI
    "P_IM_STEN",    # post-infarction angina
]

MI_MORTALITY_TARGET = "LET_IS"

mi_targets = [
    col
    for col in MI_BINARY_TARGETS + [MI_MORTALITY_TARGET]
    if col in mi.columns
]

print("Detected target columns:")
print(mi_targets)

missing_targets = [
    col
    for col in MI_BINARY_TARGETS
    if col not in mi.columns
]

if missing_targets:
    print("\nWarning - target columns not found:")
    print(missing_targets)

Detected target columns:
['FIBR_PREDS', 'PREDS_TAH', 'JELUD_TAH', 'FIBR_JELUD', 'A_V_BLOK', 'RAZRIV', 'DRESSLER', 'ZSN', 'REC_IM', 'P_IM_STEN', 'LET_IS']

Warning - target columns not found:
['OTEC_LANC']


#### FRAMINGHAM

In [13]:
framingham_files = list(DATASETS["framingham"].glob("*.csv"))

if not framingham_files:
    raise FileNotFoundError("No Framingham CSV found.")

framingham = pd.read_csv(framingham_files[0])

print("Framingham shape:", framingham.shape)
display(framingham.head())

Framingham shape: (4240, 16)


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [14]:
if "TenYearCHD" not in framingham.columns:
    raise KeyError("TenYearCHD not found.")

framingham["TenYearCHD"] = framingham["TenYearCHD"].astype(int)

print(
    framingham["TenYearCHD"]
    .value_counts()
    .sort_index()
)

TenYearCHD
0    3596
1     644
Name: count, dtype: int64


In [15]:
def split_data(df, target_columns, random_state=42):
    target_columns = [
        col for col in target_columns
        if col in df.columns
    ]

    feature_columns = [
        col for col in df.columns
        if col not in target_columns
    ]

    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        random_state=random_state,
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=random_state,
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        feature_columns,
        target_columns,
    )

In [16]:
def preprocess_tabular_dataset(
    df,
    target_columns,
    dataset_name,
):
    (
        train_df,
        val_df,
        test_df,
        feature_columns,
        target_columns,
    ) = split_data(df, target_columns)

    X_train = train_df[feature_columns].copy()
    X_val = val_df[feature_columns].copy()
    X_test = test_df[feature_columns].copy()

    y_train = train_df[target_columns].copy()
    y_val = val_df[target_columns].copy()
    y_test = test_df[target_columns].copy()

    numeric_features = X_train.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_features = [
        col for col in X_train.columns
        if col not in numeric_features
    ]

    # Numeric preprocessing
    numeric_imputer = SimpleImputer(
        strategy="median"
    )

    X_train_num = numeric_imputer.fit_transform(
        X_train[numeric_features]
    )

    X_val_num = numeric_imputer.transform(
        X_val[numeric_features]
    )

    X_test_num = numeric_imputer.transform(
        X_test[numeric_features]
    )

    scaler = StandardScaler()

    X_train_num = scaler.fit_transform(X_train_num)
    X_val_num = scaler.transform(X_val_num)
    X_test_num = scaler.transform(X_test_num)

    # Categorical preprocessing
    if categorical_features:
        categorical_imputer = SimpleImputer(
            strategy="most_frequent"
        )

        X_train_cat = categorical_imputer.fit_transform(
            X_train[categorical_features]
        )

        X_val_cat = categorical_imputer.transform(
            X_val[categorical_features]
        )

        X_test_cat = categorical_imputer.transform(
            X_test[categorical_features]
        )

        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

        X_train_cat = encoder.fit_transform(X_train_cat)
        X_val_cat = encoder.transform(X_val_cat)
        X_test_cat = encoder.transform(X_test_cat)

        X_train_processed = np.hstack([
            X_train_num,
            X_train_cat
        ])

        X_val_processed = np.hstack([
            X_val_num,
            X_val_cat
        ])

        X_test_processed = np.hstack([
            X_test_num,
            X_test_cat
        ])

    else:
        categorical_imputer = None
        encoder = None

        X_train_processed = X_train_num
        X_val_processed = X_val_num
        X_test_processed = X_test_num

    result = {
        "X_train": X_train_processed.astype(np.float32),
        "X_val": X_val_processed.astype(np.float32),
        "X_test": X_test_processed.astype(np.float32),

        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,

        "numeric_features": numeric_features,
        "categorical_features": categorical_features,

        "numeric_imputer": numeric_imputer,
        "categorical_imputer": categorical_imputer,
        "scaler": scaler,
        "encoder": encoder,
    }

    print(f"\n{dataset_name}")
    print("-" * 50)
    print("Train:", result["X_train"].shape)
    print("Val  :", result["X_val"].shape)
    print("Test :", result["X_test"].shape)

    return result

In [17]:
zheen_processed = preprocess_tabular_dataset(
    zheen,
    target_columns=["target"],
    dataset_name="Zheen",
)


Zheen
--------------------------------------------------
Train: (923, 8)
Val  : (198, 8)
Test : (198, 8)


In [18]:
uci_processed = preprocess_tabular_dataset(
    uci_hf,
    target_columns=["DEATH_EVENT"],
    dataset_name="UCI Heart Failure",
)


UCI Heart Failure
--------------------------------------------------
Train: (209, 11)
Val  : (45, 11)
Test : (45, 11)


In [19]:
mi_processed = preprocess_tabular_dataset(
    mi,
    target_columns=mi_targets,
    dataset_name="MI Complications",
)


MI Complications
--------------------------------------------------
Train: (1190, 113)
Val  : (255, 113)
Test : (255, 113)


In [20]:
framingham_processed = preprocess_tabular_dataset(
    framingham,
    target_columns=["TenYearCHD"],
    dataset_name="Framingham",
)


Framingham
--------------------------------------------------
Train: (2968, 15)
Val  : (636, 15)
Test : (636, 15)


In [21]:
def save_processed_dataset(
    result,
    dataset_name,
):
    output_dir = PROCESSED_DIR / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)

    np.save(
        output_dir / "X_train.npy",
        result["X_train"]
    )

    np.save(
        output_dir / "X_val.npy",
        result["X_val"]
    )

    np.save(
        output_dir / "X_test.npy",
        result["X_test"]
    )

    result["y_train"].to_csv(
        output_dir / "y_train.csv",
        index=False
    )

    result["y_val"].to_csv(
        output_dir / "y_val.csv",
        index=False
    )

    result["y_test"].to_csv(
        output_dir / "y_test.csv",
        index=False
    )

    metadata = {
        "dataset": dataset_name,
        "train_shape": list(result["X_train"].shape),
        "val_shape": list(result["X_val"].shape),
        "test_shape": list(result["X_test"].shape),
        "numeric_features": result["numeric_features"],
        "categorical_features": result["categorical_features"],
    }

    with open(
        output_dir / "metadata.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            metadata,
            f,
            indent=2
        )

    with open(
        output_dir / "preprocessors.pkl",
        "wb"
    ) as f:
        pickle.dump(
            {
                "numeric_imputer": result["numeric_imputer"],
                "categorical_imputer": result["categorical_imputer"],
                "scaler": result["scaler"],
                "encoder": result["encoder"],
                "numeric_features": result["numeric_features"],
                "categorical_features": result["categorical_features"],
            },
            f
        )

    print(f"Saved: {output_dir}")

In [22]:
save_processed_dataset(
    zheen_processed,
    "zheen"
)

save_processed_dataset(
    uci_processed,
    "uci_heart_failure"
)

save_processed_dataset(
    mi_processed,
    "mi_complications"
)

save_processed_dataset(
    framingham_processed,
    "framingham"
)

Saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\biomarkers\zheen
Saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\biomarkers\uci_heart_failure
Saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\biomarkers\mi_complications
Saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\biomarkers\framingham


In [23]:
processed_sets = {
    "zheen": zheen_processed,
    "uci_heart_failure": uci_processed,
    "mi_complications": mi_processed,
    "framingham": framingham_processed,
}

for name, data in processed_sets.items():

    print(f"\n{name}")

    for split in ["X_train", "X_val", "X_test"]:
        X = data[split]

        print(
            f"{split:8}: shape={X.shape}, "
            f"NaN={np.isnan(X).sum()}, "
            f"Inf={np.isinf(X).sum()}"
        )


zheen
X_train : shape=(923, 8), NaN=0, Inf=0
X_val   : shape=(198, 8), NaN=0, Inf=0
X_test  : shape=(198, 8), NaN=0, Inf=0

uci_heart_failure
X_train : shape=(209, 11), NaN=0, Inf=0
X_val   : shape=(45, 11), NaN=0, Inf=0
X_test  : shape=(45, 11), NaN=0, Inf=0

mi_complications
X_train : shape=(1190, 113), NaN=0, Inf=0
X_val   : shape=(255, 113), NaN=0, Inf=0
X_test  : shape=(255, 113), NaN=0, Inf=0

framingham
X_train : shape=(2968, 15), NaN=0, Inf=0
X_val   : shape=(636, 15), NaN=0, Inf=0
X_test  : shape=(636, 15), NaN=0, Inf=0
